In [41]:
# ============================================
# CELL 1 — Load data and basic setup
# ============================================
import pandas as pd
import numpy as np

df = pd.read_excel('../data/market_data_raw.xlsx')

# Convert date column from text to actual datetime (needed for all time-based operations)
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')

# Sort ONCE here, right after loading. This prevents index confusion later —
# every cell after this assumes data is already in ticker+date order.
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

# Work on a copy so we always have the original 'df' untouched for comparison
df_clean = df.copy()

print("Shape:", df_clean.shape)
df_clean.head()

Shape: (654, 7)


,date,ticker,sector,asset_class,price,volume,analyst
0,2024-01-01,aapl,Tech,Equity,154.61,2799407.0,GS
1,2024-01-03,aapl,Tech,Equity,153.50,1276094.0,GS
2,2024-01-04,aapl,Tech,Equity,154.67,2778829.0,JPM
3,2024-01-17,aapl,Tech,Equity,151.59,3746464.0,JPM
4,2024-01-19,aapl,Tech,Equity,150.30,5211723.0,JPM


In [42]:
# ============================================
# CELL 2 — Clean up string/category columns
# ============================================
# Fixes inconsistent casing and stray whitespace (e.g. ' aapl ' vs 'AAPL')
# This matters because groupby/value_counts treat these as DIFFERENT values otherwise
df_clean['ticker'] = df_clean['ticker'].str.upper().str.strip()
df_clean['asset_class'] = df_clean['asset_class'].str.upper().str.strip()

print(df_clean['ticker'].unique())
print(df_clean['asset_class'].unique())

['AAPL' 'JPM' 'MSFT' 'T10Y' 'XOM']
['EQUITY' 'FIXED INCOME']


In [43]:
# ============================================
# CELL 3 — Remove duplicate ticker+date records
# ============================================
# Same ticker + same date appearing twice with different prices = bad upstream join/append
# We keep the FIRST occurrence and drop the rest
print("Rows before removing duplicates:", df_clean.shape[0])

df_clean = df_clean.drop_duplicates(subset=['ticker', 'date'], keep='first').reset_index(drop=True)

print("Rows after removing duplicates:", df_clean.shape[0])

Rows before removing duplicates: 654
Rows after removing duplicates: 650


In [44]:
# ============================================
# CELL 4 — Fix negative volume
# ============================================
# Volume can never legitimately be negative — this is always a sign-flip data entry error.
# We take the absolute value to recover the likely true number.
df_clean.loc[df_clean['volume'] < 0, 'volume'] = df_clean.loc[df_clean['volume'] < 0, 'volume'].abs()

# Verify no negative volume remains
print("Any negative volume left?", (df_clean['volume'] < 0).any())

Any negative volume left? False


In [45]:
# ============================================
# CELL 5 — Fill missing price, volume, and analyst values
# ============================================
# Mark which rows had missing price BEFORE filling, so we can be transparent
# about which values are real vs. estimated
df_clean['was_filled'] = df_clean['price'].isnull()

# Forward fill price/volume PER TICKER (so AAPL's data doesn't bleed into MSFT's)
# Logic: if a price is missing (e.g. vendor feed outage), assume it held at the last known value
df_clean['price'] = df_clean.groupby('ticker')['price'].ffill()
df_clean['volume'] = df_clean.groupby('ticker')['volume'].ffill()

# Analyst is a category, not a time series — forward filling doesn't make sense here.
# Just label missing values clearly instead of guessing.
df_clean['analyst'] = df_clean['analyst'].fillna('UNKNOWN')

print("Remaining nulls:\n", df_clean.isnull().sum())

Remaining nulls:
 date           0
ticker         0
sector         0
asset_class    0
price          0
volume         0
analyst        0
was_filled     0
dtype: int64


In [46]:
# ============================================
# CELL 6 — Calculate z-score to detect price outliers
# ============================================
# Done PER TICKER because each ticker has its own normal price range
# (comparing AAPL's $150 range to T10Y's $98 range directly would be meaningless)
df_clean['ticker_mean'] = df_clean.groupby('ticker')['price'].transform('mean')
df_clean['ticker_std'] = df_clean.groupby('ticker')['price'].transform('std')
df_clean['z_score'] = (df_clean['price'] - df_clean['ticker_mean']) / df_clean['ticker_std']

# Flag anything more than 3 standard deviations away as a likely data error
df_clean['is_outlier'] = df_clean['z_score'].abs() > 3

outliers = df_clean[df_clean['is_outlier']]
print(f"Found {len(outliers)} outliers:")
outliers[['date', 'ticker', 'price', 'z_score']]

Found 2 outliers:


,date,ticker,price,z_score
67,2024-03-25,AAPL,2585.88,11.304355
220,2024-05-06,JPM,7.85,-9.688033


In [47]:
# ============================================
# CELL 7 Fix outliers using pandas built-in interpolate()
# ============================================
# Step 1: mark known outlier prices as missing (NaN) — we don't trust them anymore
df_clean.loc[df_clean['is_outlier'], 'price'] = np.nan

# Step 2: interpolate PER TICKER — fills NaN using a straight line between
# the nearest valid value before and after it (same idea as averaging neighbors)
df_clean['price'] = df_clean.groupby('ticker')['price'].transform(lambda x: x.interpolate())

print("Outliers remaining:", df_clean['price'].isnull().sum())

Outliers remaining: 0


In [48]:
# ============================================
# CELL 8 — Recalculate stats AFTER fixing outliers
# ============================================
# IMPORTANT: any time you change the 'price' column, all derived stats
# (mean, std, z-score) become stale and MUST be recalculated.
# Skipping this step is the #1 cause of "fixed data still shows as an outlier" bugs.
df_clean['ticker_mean'] = df_clean.groupby('ticker')['price'].transform('mean')
df_clean['ticker_std'] = df_clean.groupby('ticker')['price'].transform('std')
df_clean['z_score'] = (df_clean['price'] - df_clean['ticker_mean']) / df_clean['ticker_std']
df_clean['is_outlier'] = df_clean['z_score'].abs() > 3

print("Outliers remaining after fix:", df_clean['is_outlier'].sum())

Outliers remaining after fix: 0


In [49]:
# ============================================
# CELL 9 — Detect stale/frozen prices
# ============================================
# A price that's EXACTLY identical to the previous day, for the SAME ticker,
# suggests a frozen/broken data feed rather than a genuine market pause.
df_clean['price_diff'] = df_clean.groupby('ticker')['price'].diff()
df_clean['is_stale'] = df_clean['price_diff'] == 0

stale_rows = df_clean[df_clean['is_stale']]
print(f"Found {len(stale_rows)} stale price rows:")
stale_rows[['date', 'ticker', 'price', 'price_diff']]

Found 25 stale price rows:


,date,ticker,price,price_diff
12,2024-03-18,AAPL,139.93,0.0
18,2024-05-10,AAPL,150.85,0.0
50,2024-02-23,AAPL,140.98,0.0
85,2024-04-19,AAPL,143.02,0.0
124,2024-06-21,AAPL,161.81,0.0
152,2024-01-31,JPM,143.29,0.0
295,2024-02-19,MSFT,295.50,0.0
319,2024-03-22,MSFT,296.60,0.0
324,2024-03-29,MSFT,290.59,0.0
326,2024-04-02,MSFT,294.32,0.0


In [50]:
# ============================================
# CELL 10 — Check for missing trading days (coverage gaps)
# ============================================
# This checks whether any ticker is COMPLETELY ABSENT for a date,
# not just blank/null within an existing row — a different, more severe issue.
all_dates = pd.date_range(df_clean['date'].min(), df_clean['date'].max(), freq='B')

for ticker in df_clean['ticker'].unique():
    ticker_dates = df_clean[df_clean['ticker'] == ticker]['date']
    missing = all_dates.difference(ticker_dates)
    print(f"{ticker}: missing {len(missing)} trading days")

AAPL: missing 0 trading days
JPM: missing 0 trading days
MSFT: missing 0 trading days
T10Y: missing 0 trading days
XOM: missing 0 trading days


In [51]:
# ============================================
# CELL 11 — Automated validation rules
# ============================================
# These rules summarize everything we checked into pass/fail checks —
# the kind of automated checks that would run daily on new data in production
def check_rule(condition, rule_name):
    status = "✅ PASSED" if condition else "❌ FAILED"
    print(f"{status}: {rule_name}")

check_rule((df_clean['price'] > 0).all(), "All prices positive")
check_rule((df_clean['volume'] >= 0).all(), "No negative volume")
check_rule(df_clean.duplicated(subset=['ticker', 'date']).sum() == 0, "No duplicate ticker+date records")
check_rule((df_clean['z_score'].abs() <= 3).all(), "No price outliers remaining")
check_rule(set(df_clean['ticker'].unique()) == {'AAPL', 'MSFT', 'JPM', 'XOM', 'T10Y'}, "All expected tickers present")

missing_pct = df_clean.groupby('ticker')['price'].apply(lambda x: x.isnull().mean() * 100)
check_rule((missing_pct <= 5).all(), "Missing price threshold per ticker")

✅ PASSED: All prices positive
✅ PASSED: No negative volume
✅ PASSED: No duplicate ticker+date records
✅ PASSED: No price outliers remaining
✅ PASSED: All expected tickers present
✅ PASSED: Missing price threshold per ticker
